# Bias in Bios Notebook

This notebook runs the local Bias in Bios pipeline and keeps caches and outputs inside the notebook folder.

## Overview

The notebook trains or reloads a classifier, computes calibration and test probabilities, and exports CSV summaries for the pooled, group-wise, and equalized-size analyses.

## Local paths and caches

This cell sets the working directory and redirects caches to local subfolders such as `outputs/` and `cache/`.

In [ ]:
import os
from pathlib import Path

# ------------------------------------------------------------------
# Set the local project root.
# In a normal local Jupyter workflow, this is the notebook directory.
# If needed, replace Path.cwd() with a manual path.
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

# Central output directory
ROOT_DIR = PROJECT_ROOT / "outputs"

# Central cache directory
CACHE_ROOT = PROJECT_ROOT / "cache"
HF_HOME_DIR = CACHE_ROOT / "huggingface"
HF_DATASETS_CACHE_DIR = HF_HOME_DIR / "datasets"
HF_HUB_CACHE_DIR = HF_HOME_DIR / "hub"
TORCH_HOME_DIR = CACHE_ROOT / "torch"
MPLCONFIGDIR_DIR = CACHE_ROOT / "matplotlib"
PIP_CACHE_DIR = CACHE_ROOT / "pip"
WANDB_DIR = CACHE_ROOT / "wandb"
XDG_CACHE_HOME_DIR = CACHE_ROOT / "xdg"
TMP_DIR = CACHE_ROOT / "tmp"

for p in [
    ROOT_DIR,
    CACHE_ROOT,
    HF_HOME_DIR,
    HF_DATASETS_CACHE_DIR,
    HF_HUB_CACHE_DIR,
    TORCH_HOME_DIR,
    MPLCONFIGDIR_DIR,
    PIP_CACHE_DIR,
    WANDB_DIR,
    XDG_CACHE_HOME_DIR,
    TMP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# Redirect caches before importing heavy libraries.
# Prefer HF_HOME-based routing. Do not set TRANSFORMERS_CACHE because
# current Transformers warns that it is deprecated.
os.environ["HF_HOME"] = str(HF_HOME_DIR)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR_DIR)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
os.environ["WANDB_DIR"] = str(WANDB_DIR)
os.environ["WANDB_CACHE_DIR"] = str(WANDB_DIR / "cache")
os.environ["XDG_CACHE_HOME"] = str(XDG_CACHE_HOME_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ROOT_DIR:", ROOT_DIR)
print("CACHE_ROOT:", CACHE_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])
print("TORCH_HOME:", os.environ["TORCH_HOME"])
print("MPLCONFIGDIR:", os.environ["MPLCONFIGDIR"])
print("PIP_CACHE_DIR:", os.environ["PIP_CACHE_DIR"])
print("WANDB_DIR:", os.environ["WANDB_DIR"])
print("TMPDIR:", os.environ["TMPDIR"])
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])



## Package check

This notebook reports package versions and avoids automatic upgrades.

In [ ]:
from pathlib import Path

ROOT_DIR = Path(ROOT_DIR)
ROOT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT_DIR:", ROOT_DIR)
print("Outputs will be stored under:", ROOT_DIR)


## Configuration

Edit the main settings here, including run mode, seeds, alpha values, and dataset size limits.

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple, set)):
        return [to_jsonable(x) for x in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, Path):
        return str(obj)
    else:
        return obj

# ============================================================
# Core experiment switches
# ============================================================
RUN_MODE = "train_and_analyze"   # "train_and_analyze" or "analyze_only"
EXPERIMENT_MODE = "full"         # Recommended paper setting: "full"
SEEDS = [4]          # Recommended paper setting

# ============================================================
# Conformal score / nonconformity score selection
# Choose one of: "simple" (1 - p), "raps", "saps"
# ------------------------------------------------------------
# Notes:
# - These scores are computed on the (optionally) temperature-adjusted probabilities.
# - If SCORE_RANDOMIZE=False, we use u=0.5 for deterministic, reproducible runs.
# ============================================================
CONFORMAL_SCORE = "saps"   # "simple", "raps", or "saps"

SCORE_TEMPERATURE = 1.0      # temperature used *inside* the score (1.0 = no change)
SCORE_RANDOMIZE = False      # True -> u ~ Unif[0,1] per example; False -> u=0.5
SCORE_RANDOM_SEED = 0        # only used when SCORE_RANDOMIZE=True

# RAPS hyperparameters
RAPS_LAMBDA = 0.2            # lambda in the RAPS penalty
RAPS_K_REG = 5               # k_reg in the RAPS penalty

# SAPS hyperparameters
SAPS_LAMBDA = 0.2            # lambda in SAPS

# Dataset / model
DATASET_NAME = "LabHC/bias_in_bios"
MODEL_NAME = "distilbert-base-uncased"
TOP_N_OCCUPATIONS = 10           # Recommended: 10 for a lighter, cleaner subset; set None for all classes

# Text preprocessing
APPLY_CLEAN_TEXT = True
TRUNCATE_TO_400_CHARS = False
MAX_LENGTH = 160

# Training settings
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS_QUICK = 2
NUM_EPOCHS_FULL = 2
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 32
WARMUP_RATIO = 0.05
EARLY_STOPPING_PATIENCE = 2
# Keep GPU training enabled, but disable fp16 because it is the most likely cause
# of the Windows/Jupyter CUDA "unknown error" seen at epoch-end evaluation.
USE_FP16 = False

# ============================================================
# Analysis layout
# ============================================================
PRIMARY_ALPHA = 0.10
ROBUSTNESS_ALPHAS = [0.05, 0.07, 0.085, 0.10]
RUN_ALPHA_ROBUSTNESS = True
RUN_TEMPERATURE_SWEEP = True

# Controlled heterogeneity sweep:
# keep one reference group at temperature 1.0 and scale the chosen group.
TEMPERATURE_SWEEP_GROUP = 1
TEMPERATURE_SWEEP_VALUES = [1.00, 1.10, 1.25, 1.50, 1.75, 2.00]
TEMPERATURE_EPS = 1e-12

GRID_STEP = 0.001

# Quick mode limits
QUICK_LIMITS = {
    "model_train": 20000,
    "model_val": 2500,
    "calibration": 5000,
    "test": 5000,
}

# Full mode uses the full split sizes
FULL_LIMITS = {
    "model_train": None,
    "model_val": None,
    "calibration": None,
    "test": None,
}

LIMITS = QUICK_LIMITS if EXPERIMENT_MODE == "quick" else FULL_LIMITS
NUM_EPOCHS = NUM_EPOCHS_QUICK if EXPERIMENT_MODE == "quick" else NUM_EPOCHS_FULL
ALPHAS = ROBUSTNESS_ALPHAS if RUN_ALPHA_ROBUSTNESS else [PRIMARY_ALPHA]

RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"bias_in_bios_{CONFORMAL_SCORE}_{EXPERIMENT_MODE}_primary_{str(PRIMARY_ALPHA).replace('.', 'p')}_{RUN_STAMP}"
RUN_ROOT = ROOT_DIR / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_mode": RUN_MODE,
    "experiment_mode": EXPERIMENT_MODE,
    "seeds": SEEDS,
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "top_n_occupations": TOP_N_OCCUPATIONS,
    "apply_clean_text": APPLY_CLEAN_TEXT,
    "truncate_to_400_chars": TRUNCATE_TO_400_CHARS,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "use_fp16": USE_FP16,
    "primary_alpha": PRIMARY_ALPHA,
    "robustness_alphas": ROBUSTNESS_ALPHAS,
    "run_alpha_robustness": RUN_ALPHA_ROBUSTNESS,
    "run_temperature_sweep": RUN_TEMPERATURE_SWEEP,
    "temperature_sweep_group": TEMPERATURE_SWEEP_GROUP,
    "temperature_sweep_values": TEMPERATURE_SWEEP_VALUES,
    "grid_step": GRID_STEP,
    "limits": LIMITS,
}

print(json.dumps(to_jsonable(CONFIG), indent=2))
print("RUN_ROOT:", RUN_ROOT)


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_global_seed(SEEDS[0])


## Data loading and preprocessing

This section loads Bias in Bios, selects labels and groups, and applies optional text cleaning.

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_NAME)
print(raw)

profession_feature = raw["train"].features["profession"]
gender_feature = raw["train"].features["gender"]

profession_names = getattr(profession_feature, "names", None)
gender_names = getattr(gender_feature, "names", None)

if profession_names is None:
    max_prof = max(raw["train"]["profession"])
    profession_names = [str(i) for i in range(max_prof + 1)]

if gender_names is None:
    max_gender = max(raw["train"]["gender"])
    gender_names = [str(i) for i in range(max_gender + 1)]

print("Number of professions in raw dataset:", len(profession_names))
print("Gender labels:", gender_names)


In [ ]:
def clean_text_basic(text: str) -> str:
    text = str(text).replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_text_extended(text: str) -> str:
    text = str(text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", " ", text)
    text = re.sub(r"\+?\d?[\d\-() ]{7,}\d", " ", text)
    text = re.sub(r"([!?.,;:])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    if TRUNCATE_TO_400_CHARS:
        text = text[:400]
    return text


def preprocess_text(text: str) -> str:
    if APPLY_CLEAN_TEXT:
        return clean_text_extended(text)
    return clean_text_basic(text)


# Optional top-N occupation filtering using training-frequency ranking.
train_professions = np.array(raw["train"]["profession"])
train_counts = Counter(train_professions)

if TOP_N_OCCUPATIONS is None:
    selected_profession_ids = sorted(train_counts.keys())
else:
    selected_profession_ids = [
        item[0] for item in sorted(train_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:TOP_N_OCCUPATIONS]
    ]

selected_profession_ids = [int(x) for x in sorted(selected_profession_ids)]
selected_profession_names = [str(profession_names[int(idx)]) for idx in selected_profession_ids]
gender_names = [str(x) for x in gender_names]
label_map = {old_id: new_id for new_id, old_id in enumerate(selected_profession_ids)}
inv_label_map = {new_id: old_id for old_id, new_id in label_map.items()}

print("Selected professions:", len(selected_profession_ids))
print(selected_profession_names[:10], "..." if len(selected_profession_names) > 10 else "")


def keep_selected(example):
    return example["profession"] in label_map


filtered = {}
for split_name in ["train", "dev", "test"]:
    ds = raw[split_name]
    if TOP_N_OCCUPATIONS is not None:
        ds = ds.filter(
            keep_selected,
            load_from_cache_file=False,
            desc=f"Filter {split_name} to selected occupations",
        )
    filtered[split_name] = ds


def preprocess_example(example):
    return {
        "text": preprocess_text(example["hard_text"]),
        "labels": int(label_map[example["profession"]]),
        "group": int(example["gender"]),
    }


prepared = {}
for split_name, ds in filtered.items():
    prepared[split_name] = ds.map(
        preprocess_example,
        remove_columns=ds.column_names,
        load_from_cache_file=False,
        desc=f"Preprocess {split_name}",
    )


def sanity_check_prepared_split(ds, split_name):
    labels = np.asarray(ds["labels"])
    groups = np.asarray(ds["group"])

    print(
        f"[{split_name}] n={len(ds)}, label_min={labels.min()}, label_max={labels.max()}, "
        f"unique_labels={len(np.unique(labels))}, groups={np.unique(groups)}"
    )

    assert labels.min() >= 0, f"{split_name}: labels.min() < 0"
    assert labels.max() < len(selected_profession_names), (
        f"{split_name}: labels.max()={labels.max()} >= num_labels={len(selected_profession_names)}"
    )


for split_name, ds in prepared.items():
    sanity_check_prepared_split(ds, split_name)

prepared


## Data splits

The training split is divided into `model_train` and `model_val`. The official `dev` split is used for calibration, and `test` is kept for evaluation.

In [ ]:
from sklearn.model_selection import train_test_split


def stratify_key_from_arrays(labels, groups):
    return pd.Series(groups).astype(str) + "_" + pd.Series(labels).astype(str)


def choose_stratify_or_none(labels, groups):
    strata = stratify_key_from_arrays(labels, groups)
    counts = strata.value_counts()
    if (counts < 2).any():
        strata = pd.Series(labels).astype(str)
        counts = strata.value_counts()
        if (counts < 2).any():
            return None
    return strata


def select_stratified_indices(labels, groups, train_fraction=0.9, seed=0):
    idx = np.arange(len(labels))
    strata = choose_stratify_or_none(labels, groups)
    train_idx, val_idx = train_test_split(
        idx,
        train_size=train_fraction,
        random_state=seed,
        stratify=strata,
    )
    return np.sort(train_idx), np.sort(val_idx)


def stratified_subsample_indices(labels, groups, max_samples, seed=0):
    n = len(labels)
    idx = np.arange(n)
    if max_samples is None or max_samples >= n:
        return idx
    strata = choose_stratify_or_none(labels, groups)
    keep_idx, _ = train_test_split(
        idx,
        train_size=max_samples,
        random_state=seed,
        stratify=strata,
    )
    return np.sort(keep_idx)


def build_splits_for_seed(seed: int):
    train_labels = np.array(prepared["train"]["labels"])
    train_groups = np.array(prepared["train"]["group"])

    model_train_idx, model_val_idx = select_stratified_indices(
        labels=train_labels,
        groups=train_groups,
        train_fraction=0.9,
        seed=seed,
    )

    splits = {
        "model_train": prepared["train"].select(model_train_idx.tolist()),
        "model_val": prepared["train"].select(model_val_idx.tolist()),
        "calibration": prepared["dev"],
        "test": prepared["test"],
    }

    for split_name, ds in list(splits.items()):
        labels = np.array(ds["labels"])
        groups = np.array(ds["group"])
        keep_idx = stratified_subsample_indices(
            labels=labels,
            groups=groups,
            max_samples=LIMITS[split_name],
            seed=seed,
        )
        splits[split_name] = ds.select(keep_idx.tolist())
    return splits


def summarize_split(ds, split_name):
    frame = pd.DataFrame({
        "label": ds["labels"],
        "group": ds["group"],
    })
    summary = frame.groupby(["group", "label"]).size().reset_index(name="count")
    summary["group_name"] = summary["group"].map(lambda x: gender_names[x] if x < len(gender_names) else str(x))
    summary["label_name"] = summary["label"].map(
        lambda x: selected_profession_names[x] if x < len(selected_profession_names) else str(x)
    )
    summary.insert(0, "split", split_name)
    return summary


preview_splits = build_splits_for_seed(SEEDS[0])
for split_name, ds in preview_splits.items():
    print(split_name, len(ds))

preview_summary = pd.concat(
    [summarize_split(ds, split_name) for split_name, ds in preview_splits.items()],
    ignore_index=True,
)
display(preview_summary.head(20))


## Tokenization and training utilities

This section defines the tokenizer, model, and training helpers.

In [ ]:
import inspect

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


def build_tokenized_splits(splits):
    tokenized = {}
    for split_name, ds in splits.items():
        tokenized[split_name] = ds.remove_columns(['group']).map(
            tokenize_batch,
            batched=True,
            load_from_cache_file=False,
            desc=f'Tokenize {split_name}',
        )

        lengths = np.array([len(x) for x in tokenized[split_name]['input_ids']])
        print(
            f"[{split_name}] token_len_min={lengths.min()}, token_len_max={lengths.max()}, "
            f"token_len_p99={np.quantile(lengths, 0.99):.1f}"
        )
        assert lengths.max() <= MAX_LENGTH, (
            f"{split_name}: token length {lengths.max()} > MAX_LENGTH={MAX_LENGTH}"
        )
        assert lengths.max() <= 512, (
            f"{split_name}: token length {lengths.max()} > DistilBERT max position 512"
        )
    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_f1': float(f1_score(labels, preds, average='macro')),
    }


def build_training_arguments(model_dir: Path, seed: int):
    sig = inspect.signature(TrainingArguments.__init__)
    params = set(sig.parameters.keys())

    kwargs = {
        'output_dir': str(model_dir),
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'num_train_epochs': NUM_EPOCHS,
        'per_device_train_batch_size': TRAIN_BATCH_SIZE,
        'per_device_eval_batch_size': EVAL_BATCH_SIZE,
        'save_total_limit': 1,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'macro_f1',
        'greater_is_better': True,
        'logging_steps': 100,
        'seed': seed,
        'data_seed': seed,
    }

    if 'evaluation_strategy' in params:
        kwargs['evaluation_strategy'] = 'epoch'
    elif 'eval_strategy' in params:
        kwargs['eval_strategy'] = 'epoch'
    else:
        raise TypeError('This transformers version supports neither evaluation_strategy nor eval_strategy in TrainingArguments.')

    if 'save_strategy' in params:
        kwargs['save_strategy'] = 'epoch'

    if 'report_to' in params:
        kwargs['report_to'] = []

    # More stable defaults for local Windows/Jupyter GPU runs.
    if 'dataloader_num_workers' in params:
        kwargs['dataloader_num_workers'] = 0
    if 'dataloader_pin_memory' in params:
        kwargs['dataloader_pin_memory'] = False
    if 'fp16' in params:
        kwargs['fp16'] = USE_FP16
    if 'fp16_full_eval' in params:
        kwargs['fp16_full_eval'] = False
    if 'bf16' in params and not USE_FP16:
        kwargs['bf16'] = False

    return TrainingArguments(**kwargs)


def build_trainer(model, training_args, tokenized, data_collator):
    sig = inspect.signature(Trainer.__init__)
    params = set(sig.parameters.keys())

    kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': tokenized['model_train'],
        'eval_dataset': tokenized['model_val'],
        'data_collator': data_collator,
        'compute_metrics': compute_metrics,
        'callbacks': [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    }

    if 'tokenizer' in params:
        kwargs['tokenizer'] = tokenizer
    elif 'processing_class' in params:
        kwargs['processing_class'] = tokenizer

    return Trainer(**kwargs)



def train_one_seed(seed: int, splits, seed_dir: Path):
    set_global_seed(seed)

    model_dir = seed_dir / 'model'
    model_dir.mkdir(parents=True, exist_ok=True)

    tokenized = build_tokenized_splits(splits)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(selected_profession_names),
    )

    training_args = build_training_arguments(model_dir=model_dir, seed=seed)

    trainer = build_trainer(
        model=model,
        training_args=training_args,
        tokenized=tokenized,
        data_collator=data_collator,
    )

    trainer.train()
    val_metrics = trainer.evaluate(tokenized['model_val'])

    train_summary = {
        'seed': seed,
        'best_metric': None if trainer.state.best_metric is None else float(trainer.state.best_metric),
        'best_model_checkpoint': trainer.state.best_model_checkpoint,
        'validation_metrics': {k: float(v) for k, v in val_metrics.items()},
    }

    return trainer, tokenized, train_summary


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def predict_probabilities(trainer: Trainer, dataset):
    pred = trainer.predict(dataset)
    probs = softmax_np(pred.predictions)
    labels = np.array(dataset['labels'])
    return probs, labels


## Conformal utilities

This notebook supports `simple`, `raps`, and `saps` scores. Set the score and its hyperparameters in the configuration cell.

In [ ]:
# Conformal utilities and CSV-only analysis helpers


def _standardize_group_name(name: str) -> str:
    raw = str(name).strip()
    lower = raw.lower()
    if lower == "male":
        return "Male"
    if lower == "female":
        return "Female"
    return raw.title() if raw else raw


def get_group_name(g: int) -> str:
    if g < len(gender_names):
        return _standardize_group_name(gender_names[g])
    return f"Group {g}"


def _apply_temperature_global(probs: np.ndarray, T: float, eps: float = 1e-12) -> np.ndarray:
    # Apply temperature scaling to probabilities (row-wise) while staying in probability space.
    probs = np.asarray(probs, dtype=float)
    safe = np.clip(probs, eps, 1.0)
    if abs(T - 1.0) < 1e-12:
        return safe / safe.sum(axis=1, keepdims=True)
    powered = safe ** (1.0 / float(T))
    return powered / powered.sum(axis=1, keepdims=True)


def _score_method() -> str:
    return str(CONFORMAL_SCORE).strip().lower()


def _random_u(n: int) -> np.ndarray:
    # Randomization used by APS/RAPS/SAPS. If disabled, returns u=0.5 for determinism.
    if bool(SCORE_RANDOMIZE):
        rng = np.random.default_rng(int(SCORE_RANDOM_SEED))
        return rng.random(n)
    return np.full(n, 0.5, dtype=float)


def all_label_scores(probs: np.ndarray, method: str = None) -> np.ndarray:
    # Return an (n, K) matrix of nonconformity scores s(x_i, y).
    # Lower score => label is more 'conforming' for that x_i.
    # Prediction set at threshold q: { y : s(x, y) <= q }.
    method = (method or _score_method()).strip().lower()
    probs = _apply_temperature_global(probs, float(SCORE_TEMPERATURE))
    probs = np.asarray(probs, dtype=float)

    if method in ("simple", "1-p", "1p"):
        return 1.0 - probs

    n, K = probs.shape
    u = _random_u(n).reshape(-1, 1)

    # rank (1=largest prob) for each label y
    order = np.argsort(-probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(n)[:, None], order] = np.arange(1, K + 1)

    # cumulative mass ahead of each label (rho)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumsum = np.cumsum(sorted_probs, axis=1)
    rho_sorted = np.concatenate([np.zeros((n, 1)), cumsum[:, :-1]], axis=1)
    rho = rho_sorted[np.arange(n)[:, None], ranks - 1]

    if method == "raps":
        lam = float(RAPS_LAMBDA)
        k_reg = int(RAPS_K_REG)
        return rho + u * probs + lam * np.maximum(ranks - k_reg, 0)

    if method == "saps":
        lam = float(SAPS_LAMBDA)
        pmax = probs.max(axis=1, keepdims=True)
        base = pmax + lam * (ranks - 2 + u)
        top = u * probs
        return np.where(ranks == 1, top, base)

    raise ValueError(f"Unknown CONFORMAL_SCORE={method!r}. Use 'simple', 'raps', or 'saps'.")


def true_label_scores(probs: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores = all_label_scores(probs)
    return scores[np.arange(len(y)), y]


def split_conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(np.partition(scores, k - 1)[k - 1])


def pooled_threshold(scores: np.ndarray, alpha: float) -> float:
    return split_conformal_quantile(scores, alpha)


def group_thresholds(scores: np.ndarray, groups: np.ndarray, alpha: float):
    out = {}
    for g in sorted(np.unique(groups)):
        out[int(g)] = split_conformal_quantile(scores[groups == g], alpha)
    return out


def empirical_cdf_at_threshold(scores_group: np.ndarray, threshold: float) -> float:
    return float(np.mean(scores_group <= threshold))


def empirical_group_coverage(scores: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else threshold_by_group
        out[int(g)] = empirical_cdf_at_threshold(scores[mask], threshold)
    return out


def average_set_size_at_threshold(scores_all: np.ndarray, threshold: float) -> float:
    return float(np.mean((scores_all <= threshold).sum(axis=1)))


def average_group_set_size(scores_all: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else threshold_by_group
        out[int(g)] = average_set_size_at_threshold(scores_all[mask], threshold)
    return out


def group_weights(groups: np.ndarray):
    unique, counts = np.unique(groups, return_counts=True)
    weights = counts / counts.sum()
    return {int(g): float(w) for g, w in zip(unique, weights)}


def weighted_sd(values_by_group, weights_by_group):
    groups = sorted(values_by_group.keys())
    vals = np.array([values_by_group[g] for g in groups], dtype=float)
    w = np.array([weights_by_group[g] for g in groups], dtype=float)
    mean = np.sum(w * vals)
    var = np.sum(w * (vals - mean) ** 2)
    return float(np.sqrt(var))


def rms_by_group(values_by_group, weights_by_group):
    groups = sorted(values_by_group.keys())
    vals = np.array([values_by_group[g] for g in groups], dtype=float)
    w = np.array([weights_by_group[g] for g in groups], dtype=float)
    return float(np.sqrt(np.sum(w * vals ** 2)))


def size_curve_from_scores(scores_all_group: np.ndarray, grid: np.ndarray) -> np.ndarray:
    cutoffs = np.sort(scores_all_group.reshape(-1))
    n_examples = scores_all_group.shape[0]
    return np.searchsorted(cutoffs, grid, side="right") / max(n_examples, 1)


def apply_group_temperature(probs: np.ndarray, groups: np.ndarray, temperature_map: dict, eps: float = 1e-12) -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    groups = np.asarray(groups)
    adjusted = np.empty_like(probs)
    for g in np.unique(groups):
        mask = groups == g
        T = float(temperature_map[int(g)])
        safe = np.clip(probs[mask], eps, 1.0)
        if abs(T - 1.0) < 1e-12:
            adjusted[mask] = safe / safe.sum(axis=1, keepdims=True)
        else:
            powered = safe ** (1.0 / T)
            adjusted[mask] = powered / powered.sum(axis=1, keepdims=True)
    return adjusted


def finite_difference_size_slope(size_curve: np.ndarray, grid: np.ndarray, threshold: float) -> float:
    idx = int(np.argmin(np.abs(grid - threshold)))
    left = max(idx - 1, 0)
    right = min(idx + 1, len(grid) - 1)
    if right == left:
        return 0.0
    return float((size_curve[right] - size_curve[left]) / (grid[right] - grid[left]))


def _make_score_grid(max_score: float, step: float, max_points: int = 5001) -> np.ndarray:
    max_score = float(max_score)
    step = float(step)
    if max_score <= 0:
        return np.array([0.0, 1.0], dtype=float)
    n_points = int(max_score / max(step, 1e-9)) + 1
    if n_points > max_points:
        return np.linspace(0.0, max_score, num=max_points, dtype=float)
    return np.arange(0.0, max_score + step / 2.0, step, dtype=float)


def run_single_alpha_experiment(
    alpha: float,
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    g_cal: np.ndarray,
    probs_test: np.ndarray,
    y_test: np.ndarray,
    g_test: np.ndarray,
    plot_dir: Path = None,
    grid_step: float = 0.001,
    make_plots: bool = True,
    tag_prefix: str = "",
):
    scores_cal = true_label_scores(probs_cal, y_cal)
    scores_test = true_label_scores(probs_test, y_test)
    scores_all_test = all_label_scores(probs_test)

    q = pooled_threshold(scores_cal, alpha)
    q_g = group_thresholds(scores_cal, g_cal, alpha)

    p_g = group_weights(g_test)
    groups_sorted = sorted(p_g.keys())

    Fg_q = empirical_group_coverage(scores_test, g_test, q)
    eps_g = {g: Fg_q[g] - (1 - alpha) for g in groups_sorted}
    sigma_delta = weighted_sd(q_g, p_g)
    rms_cov = rms_by_group(eps_g, p_g)

    l_pooled = average_group_set_size(scores_all_test, g_test, q)
    lambda_g = average_group_set_size(scores_all_test, g_test, q_g)
    delta_l_g = {g: lambda_g[g] - l_pooled[g] for g in groups_sorted}
    rms_size = rms_by_group(delta_l_g, p_g)

    max_score_grid = float(np.max(scores_all_test))
    grid = _make_score_grid(max_score=max_score_grid, step=grid_step)
    size_curves = {g: size_curve_from_scores(scores_all_test[g_test == g], grid) for g in groups_sorted}
    size_slopes_at_qg = {
        g: finite_difference_size_slope(size_curves[g], grid, q_g[g])
        for g in groups_sorted
    }
    c_eff_proxy = float(np.sum([p_g[g] * abs(size_slopes_at_qg[g]) for g in groups_sorted]))

    Fg_qg = empirical_group_coverage(scores_test, g_test, q_g)

    lambda_target = float(sum(p_g[g] * lambda_g[g] for g in groups_sorted))
    tau_g = {}
    for g in groups_sorted:
        curve = size_curves[g]
        idx = int(np.argmin(np.abs(curve - lambda_target)))
        tau_g[g] = float(grid[idx])

    Fg_tau = empirical_group_coverage(scores_test, g_test, tau_g)
    size_equalized = average_group_set_size(scores_all_test, g_test, tau_g)
    delta_cov_g = {g: Fg_tau[g] - Fg_qg[g] for g in groups_sorted}
    sigma_lambda = weighted_sd(lambda_g, p_g)
    rms_cov_from_size = rms_by_group(delta_cov_g, p_g)

    cov_change_per_size = {}
    for g in groups_sorted:
        denom = lambda_g[g] - lambda_target
        if abs(denom) < 1e-12:
            cov_change_per_size[g] = 0.0
        else:
            cov_change_per_size[g] = delta_cov_g[g] / denom
    kappa_eff_proxy = float(np.sum([p_g[g] * abs(cov_change_per_size[g]) for g in groups_sorted]))

    group_rows = []
    for g in groups_sorted:
        row = {
            "alpha": alpha,
            "group": g,
            "group_name": get_group_name(g),
            "p_g": p_g[g],
            "q_g": q_g[g],
            "coverage_pooled": Fg_q[g],
            "epsilon_pooled": eps_g[g],
            "size_pooled": l_pooled[g],
            "coverage_groupwise": Fg_qg[g],
            "lambda_g": lambda_g[g],
            "delta_size_from_groupwise": delta_l_g[g],
            "tau_g": tau_g[g],
            "size_equalized": size_equalized[g],
            "coverage_equalized_size": Fg_tau[g],
            "delta_cov_from_equalized_size": delta_cov_g[g],
            "size_slope_at_qg": size_slopes_at_qg[g],
            "cov_change_per_size": cov_change_per_size[g],
        }
        group_rows.append(row)

    group_df = pd.DataFrame(group_rows)
    summary = {
        "alpha": alpha,
        "q_pooled": q,
        "sigma_delta": sigma_delta,
        "c_eff_proxy": c_eff_proxy,
        "c_eff_times_sigma_delta": c_eff_proxy * sigma_delta,
        "rms_cov_pooled": rms_cov,
        "lambda_target": lambda_target,
        "rms_size_from_groupwise": rms_size,
        "sigma_lambda": sigma_lambda,
        "kappa_eff_proxy": kappa_eff_proxy,
        "kappa_eff_times_sigma_lambda": kappa_eff_proxy * sigma_lambda,
        "rms_cov_from_equalized_size": rms_cov_from_size,
    }

    return {
        "summary": summary,
        "group_df": group_df,
        "scores_cal": scores_cal,
        "scores_test": scores_test,
    }


## Run one seed

This function builds the split, trains or reloads the model, saves arrays, and writes per-seed CSV summaries.

In [ ]:

def run_seed(seed: int):
    print("=" * 80)
    print(f"Running seed {seed}")
    set_global_seed(seed)

    seed_dir = RUN_ROOT / f"seed_{seed}"
    arrays_dir = seed_dir / "arrays"
    tables_dir = seed_dir / "tables"
    temp_dir = seed_dir / "temperature_sweep"
    seed_dir.mkdir(parents=True, exist_ok=True)
    arrays_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(parents=True, exist_ok=True)

    splits = build_splits_for_seed(seed)
    split_summary = pd.concat(
        [summarize_split(ds, split_name) for split_name, ds in splits.items()],
        ignore_index=True,
    )
    split_summary.to_csv(tables_dir / "split_summary.csv", index=False)

    split_meta = {
        "run_name": RUN_NAME,
        "seed": seed,
        "dataset_name": DATASET_NAME,
        "model_name": MODEL_NAME,
        "experiment_mode": EXPERIMENT_MODE,
        "selected_profession_ids_original": selected_profession_ids,
        "selected_profession_names": selected_profession_names,
        "gender_names": gender_names,
        "split_sizes": {name: len(ds) for name, ds in splits.items()},
        "limits": LIMITS,
        "primary_alpha": PRIMARY_ALPHA,
        "alphas": ALPHAS,
        "max_length": MAX_LENGTH,
        "grid_step": GRID_STEP,
        "temperature_sweep_group": TEMPERATURE_SWEEP_GROUP,
        "temperature_sweep_values": TEMPERATURE_SWEEP_VALUES,
    }
    with open(seed_dir / "split_meta.json", "w") as f:
        json.dump(to_jsonable(split_meta), f, indent=2)

    probs_cal_path = arrays_dir / "probs_cal.npy"
    probs_test_path = arrays_dir / "probs_test.npy"
    y_cal_path = arrays_dir / "y_cal.npy"
    y_test_path = arrays_dir / "y_test.npy"
    g_cal_path = arrays_dir / "g_cal.npy"
    g_test_path = arrays_dir / "g_test.npy"

    if RUN_MODE == "train_and_analyze":
        trainer, tokenized, train_summary = train_one_seed(seed=seed, splits=splits, seed_dir=seed_dir)

        with open(seed_dir / "train_summary.json", "w") as f:
            json.dump(to_jsonable(train_summary), f, indent=2)

        probs_cal, y_cal = predict_probabilities(trainer, tokenized["calibration"])
        probs_test, y_test = predict_probabilities(trainer, tokenized["test"])
        g_cal = np.array(splits["calibration"]["group"])
        g_test = np.array(splits["test"]["group"])

        np.save(probs_cal_path, probs_cal)
        np.save(probs_test_path, probs_test)
        np.save(y_cal_path, y_cal)
        np.save(y_test_path, y_test)
        np.save(g_cal_path, g_cal)
        np.save(g_test_path, g_test)

        del trainer
        del tokenized
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elif RUN_MODE == "analyze_only":
        required_paths = [probs_cal_path, probs_test_path, y_cal_path, y_test_path, g_cal_path, g_test_path]
        missing = [str(p) for p in required_paths if not p.exists()]
        if missing:
            raise FileNotFoundError(
                "RUN_MODE='analyze_only' was selected, but these files are missing:\n" + "\n".join(missing)
            )
    else:
        raise ValueError("RUN_MODE must be 'train_and_analyze' or 'analyze_only'.")

    probs_cal = np.load(probs_cal_path)
    probs_test = np.load(probs_test_path)
    y_cal = np.load(y_cal_path)
    y_test = np.load(y_test_path)
    g_cal = np.load(g_cal_path)
    g_test = np.load(g_test_path)

    cal_scores = true_label_scores(probs_cal, y_cal)
    test_scores = true_label_scores(probs_test, y_test)
    np.save(arrays_dir / "cal_scores.npy", cal_scores)
    np.save(arrays_dir / "test_scores.npy", test_scores)

    all_group_tables = []
    all_summaries = []
    for alpha in ALPHAS:
        make_plots = False
        result = run_single_alpha_experiment(
            alpha=alpha,
            probs_cal=probs_cal,
            y_cal=y_cal,
            g_cal=g_cal,
            probs_test=probs_test,
            y_test=y_test,
            g_test=g_test,
            plot_dir=None,
            grid_step=GRID_STEP,
            make_plots=make_plots,
        )
        group_df = result["group_df"].copy()
        group_df.insert(0, "seed", seed)
        all_group_tables.append(group_df)

        summary = dict(result["summary"])
        summary["seed"] = seed
        summary["is_primary_alpha"] = bool(abs(alpha - PRIMARY_ALPHA) < 1e-12)
        all_summaries.append(summary)

    group_metrics_df = pd.concat(all_group_tables, ignore_index=True)
    summary_df = pd.DataFrame(all_summaries)
    interpretation_df = summary_df.copy()
    interpretation_df["A_visible"] = interpretation_df["rms_cov_pooled"] > 0
    interpretation_df["B_visible"] = interpretation_df["rms_size_from_groupwise"] > 0
    interpretation_df["C_visible"] = interpretation_df["rms_cov_from_equalized_size"] > 0

    group_metrics_df.to_csv(tables_dir / "group_metrics_by_alpha.csv", index=False)
    summary_df.to_csv(tables_dir / "alpha_summary.csv", index=False)
    interpretation_df.to_csv(tables_dir / "interpretation_table.csv", index=False)

    primary_summary_df = summary_df[np.isclose(summary_df["alpha"], PRIMARY_ALPHA)].copy()
    primary_group_df = group_metrics_df[np.isclose(group_metrics_df["alpha"], PRIMARY_ALPHA)].copy()
    primary_summary_df.to_csv(tables_dir / "primary_alpha_summary.csv", index=False)
    primary_group_df.to_csv(tables_dir / "primary_alpha_group_metrics.csv", index=False)

    temperature_summary_df = pd.DataFrame()
    if RUN_TEMPERATURE_SWEEP:
        temp_rows = []
        for temp_value in TEMPERATURE_SWEEP_VALUES:
            temperature_map = {int(g): 1.0 for g in sorted(np.unique(g_cal))}
            temperature_map[int(TEMPERATURE_SWEEP_GROUP)] = float(temp_value)

            probs_cal_temp = apply_group_temperature(probs_cal, g_cal, temperature_map, eps=TEMPERATURE_EPS)
            probs_test_temp = apply_group_temperature(probs_test, g_test, temperature_map, eps=TEMPERATURE_EPS)

            temp_result = run_single_alpha_experiment(
                alpha=PRIMARY_ALPHA,
                probs_cal=probs_cal_temp,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test_temp,
                y_test=y_test,
                g_test=g_test,
                plot_dir=None,
                grid_step=GRID_STEP,
                make_plots=False,
                tag_prefix=f"temp_{str(temp_value).replace('.', 'p')}_",
            )
            row = dict(temp_result["summary"])
            row["seed"] = seed
            row["temperature_group"] = int(TEMPERATURE_SWEEP_GROUP)
            row["temperature_group_name"] = get_group_name(int(TEMPERATURE_SWEEP_GROUP))
            row["temperature_value"] = float(temp_value)
            temp_rows.append(row)

        temperature_summary_df = pd.DataFrame(temp_rows)
        temperature_summary_df.to_csv(tables_dir / "temperature_sweep_summary.csv", index=False)

    report = {
        "run_name": RUN_NAME,
        "seed": seed,
        "config": CONFIG,
        "alpha_summary": summary_df.to_dict(orient="records"),
        "temperature_sweep_rows": temperature_summary_df.to_dict(orient="records"),
    }
    with open(seed_dir / "experiment_report.json", "w") as f:
        json.dump(to_jsonable(report), f, indent=2)

    display(primary_summary_df)
    display(primary_group_df.head(20))
    if RUN_TEMPERATURE_SWEEP and not temperature_summary_df.empty:
        display(temperature_summary_df)

    return {
        "seed": seed,
        "summary_df": summary_df,
        "group_metrics_df": group_metrics_df,
        "primary_summary_df": primary_summary_df,
        "primary_group_df": primary_group_df,
        "temperature_summary_df": temperature_summary_df,
        "seed_dir": seed_dir,
    }


## Run all seeds

Execute the requested seeds and combine the per-seed CSV outputs.

In [ ]:

all_seed_results = []
for seed in SEEDS:
    result = run_seed(seed)
    all_seed_results.append(result)

all_summary_df = pd.concat([item["summary_df"] for item in all_seed_results], ignore_index=True)
all_group_metrics_df = pd.concat([item["group_metrics_df"] for item in all_seed_results], ignore_index=True)
all_primary_summary_df = pd.concat([item["primary_summary_df"] for item in all_seed_results], ignore_index=True)
all_primary_group_df = pd.concat([item["primary_group_df"] for item in all_seed_results], ignore_index=True)

temperature_frames = [item["temperature_summary_df"] for item in all_seed_results if len(item["temperature_summary_df"]) > 0]
all_temperature_summary_df = (
    pd.concat(temperature_frames, ignore_index=True)
    if len(temperature_frames) > 0 else
    pd.DataFrame()
)

all_summary_df.to_csv(RUN_ROOT / "all_seeds_alpha_summary.csv", index=False)
all_group_metrics_df.to_csv(RUN_ROOT / "all_seeds_group_metrics.csv", index=False)
all_primary_summary_df.to_csv(RUN_ROOT / "all_seeds_primary_alpha_summary.csv", index=False)
all_primary_group_df.to_csv(RUN_ROOT / "all_seeds_primary_alpha_group_metrics.csv", index=False)
if not all_temperature_summary_df.empty:
    all_temperature_summary_df.to_csv(RUN_ROOT / "all_seeds_temperature_sweep_summary.csv", index=False)

print("Saved combined tables to:")
print(RUN_ROOT / "all_seeds_alpha_summary.csv")
print(RUN_ROOT / "all_seeds_group_metrics.csv")
print(RUN_ROOT / "all_seeds_primary_alpha_summary.csv")
print(RUN_ROOT / "all_seeds_primary_alpha_group_metrics.csv")
if not all_temperature_summary_df.empty:
    print(RUN_ROOT / "all_seeds_temperature_sweep_summary.csv")

display(all_primary_summary_df)
display(all_primary_group_df.head(20))
if not all_temperature_summary_df.empty:
    display(all_temperature_summary_df.head(20))


## Suggested runs

Use `EXPERIMENT_MODE="quick"` with one seed for a smoke test. After that, switch to `EXPERIMENT_MODE="full"` and a larger seed list.